In [1]:
import re


def extract_times(text):
    """
    Извлекает все временные метки формата HH:MM:SS
    из произвольного текста.
    """

    pattern = r"\b\d{2}:\d{2}:\d{2}\b"

    times = re.findall(pattern, text)

    return times


In [3]:
raw_text ="""[Global] PQ QCS
100
Время выполнения и комментарий
Бонусы
Стоимость
14:44:31
20
13:48:32
20
13:20:28
20
10:41:35
20
09:31:54
20

[Global] PQ Task
3 864
Время выполнения и комментарий
Бонусы
Стоимость
15:14:54
126
15:11:05
126
15:08:04
126
14:58:18
126
14:56:24
126
14:51:24
126
14:48:11
126
14:42:51
126
13:57:15
126
13:50:49
126
13:46:48
126
13:37:20
126
13:31:45
126
13:24:34
126
13:19:57
105
11:26:26
105
11:23:02
105
11:07:15
105
10:58:41
105
10:55:18
105
10:50:41
105
10:46:06
105
10:18:43
126
10:11:59
126
10:07:55
126
10:05:57
126
09:56:36
126
09:50:12
126
09:43:35
126
09:43:04
126
09:40:15
126
09:36:57
126"""

In [4]:
text = extract_times(raw_text)

In [5]:
text

['14:44:31',
 '13:48:32',
 '13:20:28',
 '10:41:35',
 '09:31:54',
 '15:14:54',
 '15:11:05',
 '15:08:04',
 '14:58:18',
 '14:56:24',
 '14:51:24',
 '14:48:11',
 '14:42:51',
 '13:57:15',
 '13:50:49',
 '13:46:48',
 '13:37:20',
 '13:31:45',
 '13:24:34',
 '13:19:57',
 '11:26:26',
 '11:23:02',
 '11:07:15',
 '10:58:41',
 '10:55:18',
 '10:50:41',
 '10:46:06',
 '10:18:43',
 '10:11:59',
 '10:07:55',
 '10:05:57',
 '09:56:36',
 '09:50:12',
 '09:43:35',
 '09:43:04',
 '09:40:15',
 '09:36:57']

In [6]:
from datetime import datetime


def parse_time(time_str):
    return datetime.strptime(time_str, "%H:%M:%S")


def format_seconds(seconds):
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    return f"{hours} ч {minutes} мин {secs} сек"


def calculate_work_and_breaks(times, break_threshold_minutes=15):
    # Преобразуем строки времени в datetime
    parsed_times = [parse_time(t) for t in times]

    # Сортируем по времени
    parsed_times.sort()

    sessions = []
    breaks = []

    session_start = parsed_times[0]
    prev_time = parsed_times[0]

    for current_time in parsed_times[1:]:
        diff_seconds = int((current_time - prev_time).total_seconds())

        # Если перерыв больше порога — заканчиваем текущую сессию
        if diff_seconds > break_threshold_minutes * 60:
            sessions.append((session_start, prev_time))

            breaks.append({
                "start": prev_time,
                "end": current_time,
                "duration": diff_seconds
            })

            session_start = current_time

        prev_time = current_time

    # Добавляем последнюю сессию
    sessions.append((session_start, prev_time))

    # Считаем общее рабочее время
    total_work_seconds = sum(
        int((end - start).total_seconds())
        for start, end in sessions
    )

    # Считаем общее время перерывов
    total_break_seconds = sum(
        b["duration"]
        for b in breaks
    )

    # Вывод
    print("Рабочие сессии:")
    for i, (start, end) in enumerate(sessions, 1):
        duration = int((end - start).total_seconds())
        print(
            f"{i}. {start.time()} — {end.time()} "
            f"= {format_seconds(duration)}"
        )

    print("\nПерерывы:")
    for i, b in enumerate(breaks, 1):
        print(
            f"{i}. {b['start'].time()} — {b['end'].time()} "
            f"= {format_seconds(b['duration'])}"
        )

    print("\nИТОГО:")
    print("Работа:", format_seconds(total_work_seconds))
    print("Перерывы:", format_seconds(total_break_seconds))


In [10]:
calculate_work_and_breaks(text, break_threshold_minutes=16)

Рабочие сессии:
1. 09:31:54 — 10:18:43 = 0 ч 46 мин 49 сек
2. 10:41:35 — 11:26:26 = 0 ч 44 мин 51 сек
3. 13:19:57 — 13:57:15 = 0 ч 37 мин 18 сек
4. 14:42:51 — 15:14:54 = 0 ч 32 мин 3 сек

Перерывы:
1. 10:18:43 — 10:41:35 = 0 ч 22 мин 52 сек
2. 11:26:26 — 13:19:57 = 1 ч 53 мин 31 сек
3. 13:57:15 — 14:42:51 = 0 ч 45 мин 36 сек

ИТОГО:
Работа: 2 ч 41 мин 1 сек
Перерывы: 3 ч 1 мин 59 сек
